# 03 · Data structures — exercise solutions

Complete answers to all six exercises in [the student chapter](../notebooks/03_data_structures.ipynb). Run this notebook independently in a fresh Python 3 kernel. Each answer defines its own inputs and helpers; no student notebook or imported example file is required.

Try the exercises first, then compare the result and the reasoning. Assertions check values, boundary cases, and unintended changes to inputs. Helper functions organize repeated checks; an equivalent direct loop is also a valid solution.


## Solution 03.1 · Edit a queue and copy a matrix

`insert(1, ...)` puts the new person before the original second person. `pop(0)` both removes and returns the served person. `reversed` reads the current queue backward without mutating it.

Copying each row makes the matrix independent at the two levels used here. A plain `matrix.copy()` shares row objects, so changing an element through a copied row would also change the original matrix. This answer does not promise a fully deep copy of arbitrary objects inside each row.


In [ ]:
def copy_matrix_rows(matrix):
    """Copy the outer list and each row; objects inside rows remain shared."""
    return [row.copy() for row in matrix]

queue = ["Ada", "Ben", "Cara"]
queue.append("Dee")
queue.insert(1, "Eli")
queue.remove("Cara")
served = queue.pop(0)
reversed_queue = list(reversed(queue))

matrix = [[1, 2], [3, 4]]
matrix_copy = copy_matrix_rows(matrix)
matrix_copy[0][0] = 99

assert served == "Ada"
assert queue == ["Eli", "Ben", "Dee"]
assert reversed_queue == ["Dee", "Ben", "Eli"]
assert reversed_queue is not queue
assert matrix_copy == [[99, 2], [3, 4]]
assert matrix == [[1, 2], [3, 4]]
assert matrix_copy[0] is not matrix[0]
assert copy_matrix_rows([]) == []
assert copy_matrix_rows([[], [5]]) == [[], [5]]
print(served, queue, reversed_queue, matrix_copy)
print("03.1 checks passed")


## Solution 03.2 · Fibonacci records

Append the current `a` before updating `a` and `b`. Simultaneous assignment uses the old `a` and `b`, so the next pair is correct. Enumerating the finished list yields `(index, value)` tuples. The helper explicitly rejects negative counts; the exercise assumes a nonnegative integer.


In [ ]:
def fibonacci_records(n):
    """Return (index, Fibonacci number) pairs for nonnegative integer n."""
    if n < 0:
        raise ValueError("n must be nonnegative")
    values = []
    a, b = 0, 1
    for _ in range(n):
        values.append(a)
        a, b = b, a + b
    return list(enumerate(values))

records = fibonacci_records(7)
assert records == [(0, 0), (1, 1), (2, 1), (3, 2), (4, 3), (5, 5), (6, 8)]
assert fibonacci_records(0) == []
assert fibonacci_records(1) == [(0, 0)]
assert fibonacci_records(2) == [(0, 0), (1, 1)]
try:
    fibonacci_records(-1)
except ValueError:
    print("Expected ValueError: a negative count is not supported.")
else:
    raise AssertionError("A negative count should fail")
print(records)
print("03.2 checks passed")


## Solution 03.3 · Count and group names

`get(name, 0) + 1` handles both the first and later visits. `setdefault(city, [])` inserts an empty group only if needed and returns the appropriate city's list. Repeated visits remain in that list because the task asks for visits, not unique people.


In [ ]:
def summarize_visits(visits):
    """Return (counts_by_person, visitors_by_city), preserving encounter order."""
    visit_counts = {}
    people_by_city = {}
    for name, city in visits:
        visit_counts[name] = visit_counts.get(name, 0) + 1
        people_by_city.setdefault(city, []).append(name)
    return visit_counts, people_by_city

visits = [("Ada", "Vienna"), ("Bo", "Graz"), ("Ada", "Vienna"), ("Cara", "Vienna")]
original_visits = visits.copy()
visit_counts, people_by_city = summarize_visits(visits)
missing_visits = visit_counts.get("Dee", 0)
assert visit_counts == {"Ada": 2, "Bo": 1, "Cara": 1}
assert people_by_city == {"Vienna": ["Ada", "Ada", "Cara"], "Graz": ["Bo"]}
assert missing_visits == 0
assert "Dee" not in visit_counts
assert visits == original_visits
assert summarize_visits([]) == ({}, {})
assert people_by_city["Vienna"] is not people_by_city["Graz"]
assert list(people_by_city) == ["Vienna", "Graz"]
print(visit_counts, people_by_city)
print("03.3 checks passed")


## Solution 03.4 · Compare course enrollments

A set stores each distinct name once, so repeated signups disappear. Perform membership comparisons as set operations, then sort only for a reproducible presentation. Identical input memberships have an empty symmetric difference.


In [ ]:
def compare_enrollments(python_students, statistics_students):
    """Return sorted membership comparisons; repeated signups count once."""
    python_set = set(python_students)
    statistics_set = set(statistics_students)
    return {
        "both": sorted(python_set & statistics_set),
        "either": sorted(python_set | statistics_set),
        "python_only": sorted(python_set - statistics_set),
        "exactly_one": sorted(python_set ^ statistics_set),
    }

python_students = ["Ada", "Bo", "Cara", "Ada"]
statistics_students = ["Bo", "Dee"]
comparison = compare_enrollments(python_students, statistics_students)
assert comparison == {
    "both": ["Bo"],
    "either": ["Ada", "Bo", "Cara", "Dee"],
    "python_only": ["Ada", "Cara"],
    "exactly_one": ["Ada", "Cara", "Dee"],
}
assert compare_enrollments([], []) == {
    "both": [], "either": [], "python_only": [], "exactly_one": []
}
identical = compare_enrollments(["Bo", "Ada"], ["Ada", "Bo", "Ada"])
assert identical["both"] == ["Ada", "Bo"]
assert identical["python_only"] == identical["exactly_one"] == []
assert python_students == ["Ada", "Bo", "Cara", "Ada"]
print(comparison)
print("03.4 checks passed")


## Solution 03.5 · A comprehension toolkit

The list results retain duplicates and order. The dictionary collapses repeated lowercase keys, while the set collapses repeated initials. The `if word` guard is required before indexing `word[0]` because the input includes an empty string.

The length-based results use the already normalized words. This also handles Unicode text whose lowercase form has a different number of characters.


In [ ]:
def word_toolkit(words):
    """Return lowercase forms, long words, word lengths, and unique initials."""
    normalized = [word.lower() for word in words]
    long_words = [word for word in normalized if len(word) >= 5]
    lengths = {word: len(word) for word in normalized if word}
    initials = {word[0].upper() for word in words if word}
    return {
        "normalized": normalized,
        "long_words": long_words,
        "lengths": lengths,
        "initials": initials,
    }

words = ["Apple", "pear", "BANANA", "pear", "", "kiwi"]
original_words = words.copy()
results = word_toolkit(words)
assert results["normalized"] == ["apple", "pear", "banana", "pear", "", "kiwi"]
assert results["long_words"] == ["apple", "banana"]
assert results["lengths"] == {"apple": 5, "pear": 4, "banana": 6, "kiwi": 4}
assert results["initials"] == {"A", "P", "B", "K"}
assert words == original_words
assert word_toolkit([]) == {"normalized": [], "long_words": [], "lengths": {}, "initials": set()}
assert word_toolkit([""])["initials"] == set()
assert word_toolkit(["APPLE", "apple"])["lengths"] == {"apple": 5}
# Some Unicode lowercase forms contain more characters than the original.
unicode_word = "İABC"
unicode_results = word_toolkit([unicode_word])
assert unicode_results["lengths"] == {unicode_word.lower(): 5}
assert unicode_results["long_words"] == [unicode_word.lower()]
print(results["normalized"], results["lengths"], sorted(results["initials"]))
print("03.5 checks passed")


## Solution 03.6 · Build a small result table

`zip(..., strict=True)` checks paired lengths when it is consumed. The key function selects each record's score. Python's stable sort keeps Ada before Cara when their scores tie, even with `reverse=True`. The dictionary includes only records meeting the pass threshold.

The helper assumes names are distinct: using repeated names as dictionary keys would keep only the last corresponding passing entry. Its `pass_mark` setting is a preview of default parameters in the next chapter.


In [ ]:
def rank_scores(names, scores, pass_mark=70):
    """Return ranked, passed, and numbered records for distinct student names.

    Raise ValueError if the input lengths differ. Preserve input order on ties.
    Names are assumed unique because the passed result uses names as dict keys.
    """
    records = list(zip(names, scores, strict=True))
    ranked = sorted(records, key=lambda record: record[1], reverse=True)
    passed = {name: score for name, score in ranked if score >= pass_mark}
    numbered = list(enumerate(ranked, start=1))
    return {"ranked": ranked, "passed": passed, "numbered": numbered}

names = ["Ada", "Bo", "Cara"]
scores = [88, 65, 88]
results = rank_scores(names, scores)
assert results["ranked"] == [("Ada", 88), ("Cara", 88), ("Bo", 65)]
assert results["passed"] == {"Ada": 88, "Cara": 88}
assert results["numbered"] == [(1, ("Ada", 88)), (2, ("Cara", 88)), (3, ("Bo", 65))]
assert rank_scores([], []) == {"ranked": [], "passed": {}, "numbered": []}
assert rank_scores(["Threshold"], [70])["passed"] == {"Threshold": 70}
assert names == ["Ada", "Bo", "Cara"] and scores == [88, 65, 88]
for short_names, short_scores in [(["Ada"], []), ([], [90])]:
    try:
        rank_scores(short_names, short_scores)
    except ValueError:
        print("Expected ValueError: unequal input lengths.")
    else:
        raise AssertionError("A length mismatch should fail")
print(results)
print("03.6 checks passed")


All six solutions use only Python's standard features. To practice further, alter the data first and predict the resulting container types, order, and sharing before running the code again. Return to [the student chapter](../notebooks/03_data_structures.ipynb).
